# ЦПТ-фильтр: исходные и винзоризированные наблюдения Парето

Ноутбук только читает архив `saved_path_clt_averaging_winsorized/comparison.npz`. Все оценки, статусы и агрегаты должны быть заранее подготовлены скриптом эксперимента. Здесь ничего не пересчитывается и не продолжается после сохранённого отказа.

Винзоризация применяется к каждому исходному наблюдению до осреднения: $W_i=\min(X_i,U)$. Для каждой длины блока сравниваются исходная и винзоризированная ветви на их собственных достигнутых горизонтах.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

EXPERIMENT_DIR = Path('saved_path_clt_averaging_winsorized')
if not (EXPERIMENT_DIR / 'comparison.npz').exists():
    EXPERIMENT_DIR = Path('..') / EXPERIMENT_DIR
ARCHIVE_PATH = EXPERIMENT_DIR / 'comparison.npz'

with np.load(ARCHIVE_PATH, allow_pickle=False) as archive:
    data = {name: archive[name] for name in archive.files}

theta = np.asarray(data['theta'])
y = np.asarray(data['y'])
jump_times = np.asarray(data['t'])
obs_raw = np.asarray(data['obs_raw'])
obs_winsorized = np.asarray(data['obs_winsorized'])
ns = [int(value) for value in np.asarray(data['ns']).reshape(-1)]
quantile = float(np.asarray(data['quantile']).item())
threshold = float(np.asarray(data['threshold']).item())
alpha = float(np.asarray(data['alpha']).item())


def scalar_text(name):
    return str(np.asarray(data[name]).item())


def scalar_int(name):
    return int(np.asarray(data[name]).item())


def path_at(grid):
    # theta[k] и y[k] действуют до соответствующего момента скачка.
    index = np.searchsorted(jump_times, grid, side='left')
    return theta[index], y[index]


def time_for(n, values):
    # Каждая сохранённая ветвь использует только собственную длину ряда.
    return np.asarray(data[f't_net_n{n}'])[:len(values)]


print(f'Архив: {ARCHIVE_PATH.as_posix()}')
print(f'Квантиль q={quantile:g}; порог U={threshold:.6g}; alpha={alpha:g}')
print(f'Размеры блоков: {ns}')

## Статусы, покрытие и интенсивность винзоризации

Покрытие — отношение числа завершённых обновлений к числу запрошенных. Число винзоризаций относится к исходным наблюдениям, вошедшим в полные блоки. Для исходной ветви эти колонки не применимы.

In [ ]:
status_rows = []
for n in ns:
    n_blocks = len(np.asarray(data[f'obs_raw_n{n}']))
    n_source = n * n_blocks
    n_winsorized = int(np.asarray(data[f'n_winsorized_n{n}']).sum())
    winsorized_share = n_winsorized / n_source if n_source else np.nan

    for branch, branch_label in [('raw', 'исходная'), ('winsorized', 'винзоризированная')]:
        requested = scalar_int(f'steps_requested_{branch}_n{n}')
        done = scalar_int(f'steps_done_{branch}_n{n}')
        failure_step = scalar_int(f'failure_step_{branch}_n{n}')
        status_rows.append({
            'ветвь': branch_label,
            'n': n,
            'статус': scalar_text(f'status_{branch}_n{n}'),
            'готово / запрошено': f'{done} / {requested}',
            'покрытие': done / requested if requested else np.nan,
            'шаг отказа': failure_step if failure_step >= 0 else '—',
            'винзоризаций': n_winsorized if branch == 'winsorized' else '—',
            'доля винзоризаций': winsorized_share if branch == 'winsorized' else np.nan,
        })

status_table = pd.DataFrame(status_rows)
display(
    status_table.style
    .format({'покрытие': '{:.2%}', 'доля винзоризаций': '{:.4%}'}, na_rep='—')
    .hide(axis='index')
)

## Окно исходных наблюдений около отказа

Показывается первое сохранённое падение исходной ветви. Горизонтальная линия — общий порог $U(q)$; точки выше него в винзоризированном потоке заменяются на $U$. Номер update переводится в индекс наблюдения как `failure_step - 1`.

In [ ]:
raw_failures = [
    (n, scalar_int(f'failure_step_raw_n{n}'))
    for n in ns
    if scalar_int(f'failure_step_raw_n{n}') >= 0
]

if not raw_failures:
    print('В архиве нет отказа исходной ветви.')
else:
    n_failure, failure_step = min(raw_failures, key=lambda item: item[1] * item[0])
    failure_index = failure_step * n_failure - 1
    radius = 12
    left = max(0, failure_index - radius)
    right = min(len(obs_raw), failure_index + radius + 1)
    update_numbers = np.arange(left, right) + 1
    raw_window = np.asarray(obs_raw[left:right]).reshape(right - left, -1)[:, 0]
    winsorized_window = np.asarray(obs_winsorized[left:right]).reshape(right - left, -1)[:, 0]

    fig, ax = plt.subplots(figsize=(11, 4), layout='constrained')
    ax.plot(update_numbers, raw_window, 'o-', ms=4, lw=1.2, color='tab:red', label='исходные')
    ax.plot(update_numbers, winsorized_window, 'o-', ms=3, lw=1.2, color='tab:blue', label='винзоризированные')
    ax.axhline(threshold, color='black', ls='--', lw=1.3, label=f'U={threshold:.4g}')
    ax.axvline(failure_index + 1, color='tab:red', ls=':', lw=1.2, label='наблюдение у отказа')
    ax.set(xlabel='номер исходного наблюдения', ylabel='значение',
           title=f'Окно около raw failure: n={n_failure}, update={failure_step}')
    ax.legend(frameon=False)
    plt.show()

    print(scalar_text(f'failure_message_raw_n{n_failure}'))

## Оценки $\theta$ и $Y$

Каждая линия строится по собственной фактически сохранённой длине. Поэтому исходная линия естественно заканчивается в месте отказа, а завершённая винзоризированная ветвь не обрезается до этого момента.

In [ ]:
n_states = int(np.asarray(data['theta_est_exact']).shape[1])
state_colors = plt.cm.tab10(np.linspace(0, 1, n_states))

for n in ns:
    raw = np.asarray(data[f'theta_est_raw_n{n}'])
    winsorized = np.asarray(data[f'theta_est_winsorized_n{n}'])
    t_full = np.asarray(data[f't_net_n{n}'])
    true_state, _ = path_at(t_full)

    fig, axes = plt.subplots(n_states, 1, figsize=(12, 2.1 * n_states), sharex=True, layout='constrained')
    axes = np.atleast_1d(axes)
    for state, ax in enumerate(axes):
        ax.step(t_full / 3600, (true_state == state).astype(float), where='post',
                color='black', lw=1.0, alpha=.65, label='истина')
        ax.plot(time_for(n, raw) / 3600, raw[:, state], color=state_colors[state],
                lw=1.1, alpha=.8, label='исходная')
        ax.plot(time_for(n, winsorized) / 3600, winsorized[:, state], color=state_colors[state],
                lw=1.5, ls='--', label='винзоризированная')
        ax.set(ylabel=fr'$P(\theta={state})$', ylim=(-.03, 1.03))
    axes[0].set_title(f'Вероятности состояний, n={n}')
    axes[-1].set_xlabel('время, часы')
    axes[0].legend(ncol=3, frameon=False)
    plt.show()

In [ ]:
n_components = int(np.asarray(data['y_est_exact']).shape[1])

for n in ns:
    raw = np.asarray(data[f'y_est_raw_n{n}'])
    winsorized = np.asarray(data[f'y_est_winsorized_n{n}'])
    t_full = np.asarray(data[f't_net_n{n}'])
    _, true_y = path_at(t_full)

    fig, axes = plt.subplots(n_components, 1, figsize=(12, 2.7 * n_components), sharex=True, layout='constrained')
    axes = np.atleast_1d(axes)
    for component, ax in enumerate(axes):
        ax.step(t_full / 3600, true_y[:, component], where='post', color='black',
                lw=1.1, alpha=.7, label='истина')
        ax.plot(time_for(n, raw) / 3600, raw[:, component], color='tab:red',
                lw=1.1, alpha=.8, label='исходная')
        ax.plot(time_for(n, winsorized) / 3600, winsorized[:, component], color='tab:blue',
                lw=1.4, ls='--', label='винзоризированная')
        ax.set_ylabel(fr'$Y_{component + 1}$')
    axes[0].set_title(f'Непрерывные компоненты, n={n}')
    axes[-1].set_xlabel('время, часы')
    axes[0].legend(ncol=3, frameon=False)
    plt.show()

## RMSE относительно скрытой траектории

Для исходной ветви RMSE считается только на реально достигнутом префиксе. Для винзоризированной ветви полный RMSE показывается лишь при статусе `completed`; незавершённые ветви намеренно не получают итоговую метрику.

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


rmse_rows = []
for n in ns:
    for branch, branch_label in [('raw', 'исходная'), ('winsorized', 'винзоризированная')]:
        status = scalar_text(f'status_{branch}_n{n}')
        theta_est = np.asarray(data[f'theta_est_{branch}_n{n}'])
        y_est = np.asarray(data[f'y_est_{branch}_n{n}'])
        show_metric = branch == 'raw' or status == 'completed'

        if show_metric:
            t_theta = time_for(n, theta_est)
            true_state, _ = path_at(t_theta)
            true_onehot = np.eye(n_states)[true_state]
            t_y = time_for(n, y_est)
            _, true_y = path_at(t_y)
            theta_rmse = rmse(theta_est, true_onehot)
            y_rmse = [rmse(y_est[:, j], true_y[:, j]) for j in range(n_components)]
        else:
            theta_rmse = np.nan
            y_rmse = [np.nan] * n_components

        row = {
            'ветвь': branch_label,
            'n': n,
            'статус': status,
            'горизонт RMSE': 'достигнутый префикс' if branch == 'raw' else ('полный' if show_metric else '—'),
            'RMSE theta': theta_rmse,
        }
        row.update({f'RMSE Y{j + 1}': value for j, value in enumerate(y_rmse)})
        rmse_rows.append(row)

rmse_table = pd.DataFrame(rmse_rows)
metric_columns = ['RMSE theta'] + [f'RMSE Y{j + 1}' for j in range(n_components)]
display(rmse_table.style.format({column: '{:.4f}' for column in metric_columns}, na_rep='—').hide(axis='index'))

## Оговорки

- Это сравнение выполнено на одной сохранённой скрытой траектории и одном потоке наблюдений; оно не оценивает распределение качества по повторным симуляциям.
- Ветви могут иметь разные достигнутые горизонты. Сравнение полного винзоризированного RMSE с префиксным raw RMSE не является сравнением на одном и том же интервале.
- Винзоризация меняет модель наблюдения: крайние значения не удаляются, а заменяются порогом $U$. Число затронутых наблюдений следует читать вместе с качеством оценок.
- Статус `completed` и отсутствие численного отказа сами по себе не доказывают корректность нормальной аппроксимации или оптимальность выбранного квантиля.